In [1]:
!git clone https://github.com/bhujbalatharva252-sketch/Recommendation-Systems-benchmarking

Cloning into 'Recommendation-Systems-benchmarking'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 80 (delta 9), reused 16 (delta 2), pack-reused 56 (from 1)
Receiving objects: 100% (80/80), 43.01 MiB | 9.69 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Updating files: 100% (38/38), done.


In [4]:
%cd Recommendation-Systems-benchmarking


/content/Recommendation-Systems-benchmarking


In [2]:
!pip install transformers accelerate bitsandbytes

import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.5 MB/s eta 0:00:00


In [5]:
train = pd.read_csv("data/processed/train.csv")

movies = pd.read_csv(
    "data/raw/movies.dat",
    sep="::",
    engine="python",
    names=["MovieID","Title","Genres"],
    encoding="latin-1"
)

# Create Prompt

In [8]:
def create_prompt(user_id,train,movies,candidate_movies):

    # Get movies watched by this user from training data
    user_history=train[train["UserID"]==user_id]

    # If user has no history, cannot create recommendation prompt
    if user_history.empty:
        return None

    # Select user's highest rated movies
    user_history=user_history.sort_values(by="Rating",ascending=False).head(20)

    # Create text description of user's preferences
    history_text=""
    for index,row in user_history.iterrows():
        movie_id=row["MovieID"]
        # Get movie information
        movie_info=movies[movies["MovieID"]== movie_id]
        if not movie_info.empty:
            title = movie_info.iloc[0]["Title"]
            genre = movie_info.iloc[0]["Genres"]
            history_text+= (
                f"{title} "
                f"({genre}) "
                f"Rating: {row['Rating']}\n")

    # Create candidate movie list
    candidate_text=""

    for index,row in candidate_movies.iterrows():
        candidate_text += (
            f"MovieID: {row['MovieID']} | "
            f"Title: {row['Title']} | "
            f"Genre: {row['Genres']}\n"
        )

    # Final prompt for LLM
    prompt= f"""

You are a movie recommendation system.
Your task is to rank movies based on the user's previous preferences.

User's watched movies:
{history_text}

Candidate movies:
{candidate_text}

Instructions:
1. Select only movies from the candidate list.
2. Rank the top 10 movies.
3. Do not recommend any movie outside the candidate list.
4. Return only the ranking.

Output format:

1. MovieID:
2. MovieID:
3. MovieID:
4. MovieID:
5. MovieID:
6. MovieID:
7. MovieID:
8. MovieID:
9. MovieID:
10. MovieID:

"""
    return prompt

# Example

In [9]:
candidate_movies = movies.sample(
    30,
    random_state=42
)

prompt = create_prompt(
    user_id=1,
    train=train,
    movies=movies,
    candidate_movies=candidate_movies
)

print(prompt)



You are a movie recommendation system.
Your task is to rank movies based on the user's previous preferences.

User's watched movies:
Back to the Future (1985) (Comedy|Sci-Fi) Rating: 5
Cinderella (1950) (Animation|Children's|Musical) Rating: 5
Christmas Story, A (1983) (Comedy|Drama) Rating: 5
Last Days of Disco, The (1998) (Drama) Rating: 5
Saving Private Ryan (1998) (Action|Drama|War) Rating: 5
Awakenings (1990) (Drama) Rating: 5
One Flew Over the Cuckoo's Nest (1975) (Drama) Rating: 5
Rain Man (1988) (Drama) Rating: 5
Sound of Music, The (1965) (Musical) Rating: 5
Ben-Hur (1959) (Action|Adventure|Drama) Rating: 5
Apollo 13 (1995) (Drama) Rating: 5
Mary Poppins (1964) (Children's|Comedy|Musical) Rating: 5
Dumbo (1941) (Animation|Children's|Musical) Rating: 5
Schindler's List (1993) (Drama|War) Rating: 5
Beauty and the Beast (1991) (Animation|Children's|Musical) Rating: 5
Bug's Life, A (1998) (Animation|Children's|Comedy) Rating: 5
Toy Story (1995) (Animation|Children's|Comedy) Rati

## LLM Prompt Design

The prompt converts user history and candidate movies into a structured natural language input for the LLM. The user history contains highly rated movies, while candidate movies are provided to restrict the model output and avoid recommending unavailable items.